<a href="https://colab.research.google.com/github/enzoyoshio/Kagoshima-daigaku-nlp-100/blob/main/chapter4/nlp100knock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第4章: 言語解析

問題30から問題35までは、以下の文章`text`（太宰治の『走れメロス』の冒頭部分）に対して、言語解析を実施せよ。問題36から問題39までは、国家を説明した文書群（日本語版ウィキペディア記事から抽出したテキスト群）をコーパスとして、言語解析を実施せよ。

In [ ]:
!pip install unidic-lite
!pip install mecab-python3

In [ ]:
text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""
text = text.replace("\n", "")

## 30. 動詞
文章`text`に含まれる動詞をすべて表示せよ。

## 30. Verbs
Display all verbs contained in the text `text`.


In [ ]:
import MeCab
tagger = MeCab.Tagger()
nodes = tagger.parseToNode(text)

while nodes:
  if nodes.feature.split(',')[0] == '動詞':
    print(nodes.surface)
  # print(nodes.feature)
  nodes = nodes.next

し
除か
なら
し
わから
ある
吹き
遊ん
暮し
来
対し
あっ


## 31. 動詞の原型
文章`text`に含まれる動詞と、その原型をすべて表示せよ。

31. Verb Forms

List all the verbs in the text and their base forms.

In [ ]:
nodes = tagger.parseToNode(text)

while nodes:
  if nodes.feature.split(',')[0] == '動詞':
    print(nodes.surface, nodes.feature.split(',')[7])
    # 10th position has a no kanji form, in 7 apparently is the kanji
  nodes = nodes.next

し 為る
除か 除く
なら 成る
し 為る
わから 分かる
ある 有る
吹き 吹く
遊ん 遊ぶ
暮し 暮らす
来 来る
対し 対する
あっ 有る


## 32. 「AのB」
文章`text`において、2つの名詞が「の」で連結されている名詞句をすべて抽出せよ。.

## 32. “A's B”
In the text `text`, extract all noun phrases in which two nouns are connected by “no”.

In [ ]:
nodes = tagger.parseToNode(text)

while nodes:

  # array = nodes.feature.split(',')

  if nodes.feature.split(',')[0] == '名詞':
    # RTE? Or in this text it never ends or almos end in nouns, or I just dont know Japanese enough lol
    next1 = nodes.next
    next2 = next1.next

    if next1.surface == 'の' and next2.feature.split(',')[0] == '名詞':
      print(nodes.surface, '-', next1.surface, '-', next2.surface)

  nodes = nodes.next


暴虐 - の - 王
村 - の - 牧人


## 33. 係り受け解析

文章`text`に係り受け解析を適用し、係り元と係り先のトークン（形態素や文節などの単位）をタブ区切り形式ですべて抽出せよ。

## 33. Análise de relações sintáticas

Aplique a análise de relações sintáticas ao texto `text` e extraia todos os tokens (unidades como morfemas ou sintagmas) que atuam como referentes e referidos, separando-os por tabulações.

In [ ]:

!pip install -U spacy ginza ja_ginza

In [ ]:
# ask arimori what is this supposed to do

import spacy
import ja_ginza

# nlp = spacy.load("ja_ginza")
nlp = spacy.load("ja_ginza", config={"components.compound_splitter.split_mode": "A"})
doc = nlp(text)

for token in doc:
  print(f'{token.text=}, {token.pos_=}, {token.dep_=}, {token.head=}')

3.8.14
5.2.0
token.text='メロス', token.pos_='PROPN', token.dep_='nsubj', token.head=激怒
token.text='は', token.pos_='ADP', token.dep_='case', token.head=メロス
token.text='激怒', token.pos_='VERB', token.dep_='ROOT', token.head=激怒
token.text='し', token.pos_='AUX', token.dep_='aux', token.head=激怒
token.text='た', token.pos_='AUX', token.dep_='aux', token.head=激怒
token.text='。', token.pos_='PUNCT', token.dep_='punct', token.head=激怒
token.text='必ず', token.pos_='ADV', token.dep_='advmod', token.head=除か
token.text='、', token.pos_='PUNCT', token.dep_='punct', token.head=必ず
token.text='かの', token.pos_='DET', token.dep_='det', token.head=暴虐
token.text='邪智', token.pos_='VERB', token.dep_='compound', token.head=暴虐
token.text='暴虐', token.pos_='NOUN', token.dep_='nmod', token.head=王
token.text='の', token.pos_='ADP', token.dep_='case', token.head=暴虐
token.text='王', token.pos_='NOUN', token.dep_='obj', token.head=除か
token.text='を', token.pos_='ADP', token.dep_='case', token.head=王
token.text='除か', token.pos_=

## 34. 主述の関係
文章`text`において、「メロス」が主語であるときの述語を抽出せよ。

## 34. Relação entre sujeito e predicado

No texto, extraia os predicados quando “Meros” for o sujeito.

In [ ]:
# probably with explanation above it helps me
# but apparently is a graph with relations
# the code below checks if the current token is MEROSU (in katakana)
# and if dependency is nsubj wich I think it is noun subject
# and then it prints token head
for token in doc:
  if token.text == 'メロス' and token.dep_ == "nsubj":
    print(token.head)

激怒
牧人


In [ ]:
from spacy import displacy

displacy.render(doc, style="dep", jupyter=True)

## 35. 係り受け木
「メロスは激怒した。」の係り受け木を可視化せよ。

## 35. Subordinado

Visualize o subordinado da frase “Meros ficou furioso”.

In [ ]:
# ??? era so visualizar? fiz antes. kakunin suru

## 36. 単語の出現頻度

問題36から39までは、Wikipediaの記事を以下のフォーマットで書き出したファイル[jawiki-country.json.gz](/data/jawiki-country.json.gz)をコーパスと見なし、統計的な分析を行う。

* 1行に1記事の情報がJSON形式で格納される
* 各行には記事名が"title"キーに、記事本文が"text"キーの辞書オブジェクトに格納され、そのオブジェクトがJSON形式で書き出される
* ファイル全体はgzipで圧縮される

まず、第3章の処理内容を参考に、Wikipedia記事からマークアップを除去し、各記事のテキストを抽出せよ。そして、コーパスにおける単語（形態素）の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 36. Frequência de ocorrência de palavras

Nas questões 36 a 39, utilizaremos como corpus o arquivo jawiki-country.json.gz, que contém artigos da Wikipédia exportados no formato a seguir, para realizar uma análise estatística.

    As informações de um artigo são armazenadas em cada linha no formato JSON
    Em cada linha, o nome do artigo é armazenado na chave “title” e o corpo do artigo na chave “text” de um objeto dicionário, e esse objeto é exportado no formato JSON
    O arquivo inteiro é compactado com gzip

Primeiramente, com base no conteúdo do Capítulo 3, remova a marcação dos artigos da Wikipédia e extraia o texto de cada artigo. Em seguida, calcule a frequência de ocorrência das palavras (morfemas) no corpus e exiba as 20 palavras mais frequentes, juntamente com sua frequência de ocorrência.

In [ ]:
!wget https://nlp100.github.io/2025/_downloads/ca47e7baf341469cd7b585f97496c020/jawiki-country.json.gz
!gunzip jawiki-country.json.gz

--2026-04-27 08:37:35--  https://nlp100.github.io/2025/_downloads/ca47e7baf341469cd7b585f97496c020/jawiki-country.json.gz
Resolving nlp100.github.io (nlp100.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to nlp100.github.io (nlp100.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5068362 (4.8M) [application/gzip]
Saving to: ‘jawiki-country.json.gz’

jawiki-country.json 100%[===================>]   4.83M  --.-KB/s    in 0.06s   

2026-04-27 08:37:35 (83.3 MB/s) - ‘jawiki-country.json.gz’ saved [5068362/5068362]



## 37. 名詞の出現頻度
コーパスにおける名詞の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 38. TF・IDF
日本に関する記事における名詞のTF・IDFスコアを求め、TF・IDFスコア上位20語とそのTF, IDF, TF・IDFを表示せよ。

## 39. Zipfの法則
コーパスにおける単語の出現頻度順位を横軸、その出現頻度を縦軸として、両対数グラフをプロットせよ。